# Intraday setups: pre-registration (Phase 10, step 1 — Gate 3)

This notebook contains **only a pre-registration**: two long-only setups for 1-hour holds inside the trading window (hour_ny 9 and 10, New York time), the development-period quantiles used for their thresholds, and the evaluation plan.

**No outcome has been computed.** No forward return, outcome statistic, or relationship between any feature and future price direction appears in this notebook or was computed before writing it. The only numbers computed here are quantiles of two features (`rel_volume_1h`, `rv_4h`) over the 2017–2023 development period. Nothing is evaluated until the pre-registration is approved.

**What the reasons may draw on.** Phases 8–9 measured volatility, volume and timing only:
- 1-hour moves are largest around the US market open. In 2024 onward the typical 1-hour move was 0.817% at 10:00 and 0.713% at 09:00 New York time, against 0.502% across all hours; the cost ratio at 10:00 New York was 0.367 (notebook 12).
- Cost ratios were lower after high-volatility days and in high-volatility years (notebooks 10–11).
- Features are complete from 2024 onward, and no pair has a correlation above 0.8 (notebook 12).

No price-direction result exists for intraday data, so the reasons below rest on these descriptive facts and on general market reasoning.

## Common rules (all setups)

**Instrument and holding period.** BTC/USDT spot, long only, 1-hour holds.

**Window hours.** A window hour is a clock hour whose first 15-minute candle has `hour_ny` 9 or 10 and opens on the hour. Its open time T is 09:00 or 10:00 New York time (13:00/14:00 UTC during US daylight saving time, 14:00/15:00 UTC otherwise).

**Timing.**
- The decision is made at T, the open of the window hour's first 15-minute candle.
- The decision uses only the features of the 15-minute candle that opened at T − 15 minutes, which closed exactly at T. By the timing rule of `add_intraday_features`, that row uses only candles that opened at or before T − 15 minutes. The daily labels on that row (`prev_day_trend`, `prev_day_vol_regime`) belong to the previous UTC day, which closed before T.
- Entry: at the open of the 15-minute candle opening at T.
- Exit: exactly 1 hour later, at the open of the 15-minute candle opening at T + 1 hour.
- At most one trade per window hour. The 09:00 and 10:00 hours of the same day are separate opportunities, and their trades do not overlap (09:00–10:00 and 10:00–11:00).

**Missing data.** A window hour is not traded if the decision candle (T − 15 min), the entry candle (T) or the exit candle (T + 1h) is missing, or if any feature a setup uses is NaN on the decision row. Such hours are counted and reported in the evaluation, never filled.

**Costs.** 0.30% round trip: 0.15% per side (0.10% fee + 0.05% slippage). The net log return of a trade is log(exit × (1 − 0.0015) / (entry × (1 + 0.0015))), the same cost convention as `run_backtest`.

**Thresholds.** A threshold that comes from a feature's distribution is a quantile of that feature over the **decision rows of development-period window hours** (T before 2024-01-01). This is the population the rule is applied to. The quantiles are computed and printed below. Signs (> 0, < 0) and the daily labels need no estimation.

## Development-period quantiles used for thresholds

Decision rows: the T − 15 min candle of every window hour with T before 2024-01-01. Only `rel_volume_1h` and `rv_4h` are summarized. No return, outcome or count of qualifying setups is computed.

In [1]:
from pathlib import Path

import pandas as pd

from cryptosense.data import load_processed
from cryptosense.features import add_features
from cryptosense.regimes import add_regimes
from cryptosense.intraday import add_intraday_features

DATA_DIR = Path.cwd().parent / "data"
SYMBOL = "BTC/USDT"
WINDOW_HOURS_NY = [9, 10]
DEV_END = pd.Timestamp("2024-01-01", tz="UTC")   # development: T before this
DECISION_LAG = pd.Timedelta("15min")
QUANTILES = [0.01, 0.25, 0.50, 0.75, 0.99]

m15 = load_processed(SYMBOL, DATA_DIR, timeframe="15m")
daily = add_regimes(add_features(load_processed(SYMBOL, DATA_DIR)))
feats = add_intraday_features(m15, daily)

window_starts = feats.index[feats["hour_ny"].isin(WINDOW_HOURS_NY) & (feats.index.minute == 0)]
dev_starts = window_starts[window_starts < DEV_END]
decision_rows = feats.reindex(dev_starts - DECISION_LAG)

print(f"development window hours (T before {DEV_END.date()}): {len(dev_starts):,}, "
      f"{dev_starts[0]} -> {dev_starts[-1]}")
print(f"decision rows present: {int(decision_rows['close'].notna().sum()):,}")

dev_quantiles = pd.DataFrame({
    col: {"non-NaN rows": int(decision_rows[col].notna().sum()),
          **{f"q{int(q * 100):02d}": decision_rows[col].quantile(q) for q in QUANTILES}}
    for col in ["rel_volume_1h", "rv_4h"]
}).T
print(dev_quantiles.to_string(formatters={
    "non-NaN rows": lambda v: f"{v:,.0f}",
    **{f"q{int(q * 100):02d}": (lambda v: f"{v:.6f}") for q in QUANTILES},
}))

development window hours (T before 2024-01-01): 4,647, 2017-08-17 13:00:00+00:00 -> 2023-12-31 15:00:00+00:00
decision rows present: 4,645
              non-NaN rows      q01      q25      q50      q75      q99
rel_volume_1h        4,404 0.209980 0.587525 0.861742 1.277251 4.241106
rv_4h                4,633 0.001920 0.006370 0.010062 0.016343 0.063280


In [2]:
# Thresholds fixed from development-period quantiles only.
REL_VOLUME_THRESHOLD = dev_quantiles.loc["rel_volume_1h", "q75"]   # S1
RV_4H_THRESHOLD = dev_quantiles.loc["rv_4h", "q50"]                # S2

print(f"S1: rel_volume_1h > {REL_VOLUME_THRESHOLD:.6f}  (75th percentile, development decision rows)")
print(f"S2: rv_4h         > {RV_4H_THRESHOLD:.6f}  (50th percentile, development decision rows)")

S1: rel_volume_1h > 1.277251  (75th percentile, development decision rows)
S2: rv_4h         > 0.010062  (50th percentile, development decision rows)


## Setup S1 — Volume-confirmed push into the open

**Idea.** In a daily uptrend, when the hour before the window rose on clearly above-normal volume, buy at the window-hour open and hold for 1 hour.

**Why it might work.** The US open brings a large wave of new participants. Phases 8–9 show this is when 1-hour moves are largest relative to costs, so a trade here has the most room to cover the 0.30% cost. General market reasoning suggests:
- a move made on unusually high participation reflects real order flow rather than noise;
- that flow may continue into the open as US participants act on the same information;
- restricting to days after a daily uptrend keeps long trades on the side of the prevailing trend.

This is a hypothesis; no intraday price-direction result supports or contradicts it.

**Conditions** (all on the decision row, the candle opening at T − 15 min):
1. `prev_day_trend == "up"`
2. `ret_1h > 0`: the close at T was above the close 1 hour earlier.
3. `rel_volume_1h > REL_VOLUME_THRESHOLD`: the 75th percentile printed above.

**Threshold justification.**
- `ret_1h > 0` is a sign condition, so nothing is estimated.
- The 75th percentile defines "clearly above normal" as the top quarter of development-period decision rows. `rel_volume_1h` already compares volume with the same clock time on the previous 20 days, so the threshold does not depend on the hour.
- The threshold was chosen as a conventional cut (top quartile) before any outcome was computed. No other quantile was tried.

## Setup S2 — Uptrend dip into an active open

**Idea.** In a daily uptrend, when price fell over the previous 4 hours while the market was active, buy at the window-hour open and hold for 1 hour.

**Why it might work.** General market reasoning suggests:
- in an established uptrend, short-term declines driven by thin overnight or European liquidity may be absorbed when US liquidity arrives at the open ("buying the dip");
- requiring above-median recent volatility selects active markets. Phases 8–9 found typical moves larger relative to costs in higher-volatility conditions, so these trades have more room to cover the 0.30% cost.

This is a hypothesis; no intraday price-direction result supports or contradicts it.

**Conditions** (all on the decision row, the candle opening at T − 15 min):
1. `prev_day_trend == "up"`
2. `ret_4h < 0`: the close at T was below the close 4 hours earlier.
3. `rv_4h > RV_4H_THRESHOLD`: the 50th percentile printed above.

**Threshold justification.**
- `ret_4h < 0` is a sign condition, so nothing is estimated.
- The median splits development-period decision rows into more active and less active halves, the simplest split that needs no further choice. No other quantile was tried.

**Overlap.** S1 and S2 can fire in the same window hour (for example `ret_1h > 0` while `ret_4h < 0`). Each setup is evaluated separately; they are not combined.

## Evaluation plan (not run)

Nothing below is computed until this pre-registration is approved. After approval, the development evaluation is run first. The held-out test is then run **once**, with exactly these rules, thresholds, measures and baselines, and is never re-run after changes.

**Periods**
- Development: window hours with T on or before 2023-12-31. All trades exit by 2023-12-31 16:00 UTC.
- Held-out test: window hours with T on or after 2024-01-01.
- The holding period is 1 hour and every window hour ends by 16:00 UTC, so no trade crosses the boundary. No purge is needed beyond assigning each trade to the period of its T.
- Features near the start of the test period may use up to 20 days of development-period candles. This is past data, available at decision time.
- Thresholds stay at the development values above for the test period.

**Measures** (per setup and period, on the net 1-hour log return after 0.30% round-trip costs)
- number of trades, and the number of window hours not traded because of missing data;
- mean;
- median;
- share of trades with a positive net return;
- standard error of the mean (sample standard deviation / √n).
- Trades are also listed separately for hour_ny 9 and hour_ny 10, with the same measures (descriptive, not counted as extra comparisons).

**Base rates** (same period, same cost, every eligible window hour treated as a trade). Each is compared with the setup mean as (setup mean − base-rate mean) / setup standard error.
1. **All window hours:** the mean over every eligible window hour (hour_ny 9 and 10).
2. **Same window hour:** the mean for hour_ny 9 and for hour_ny 10, combined with weights equal to the setup's share of trades in each hour.
3. **Same window hour and same previous-day volatility regime:** the mean in each (hour_ny, `prev_day_vol_regime`) cell, combined with weights equal to the setup's share of trades in each cell. Window hours without a `prev_day_vol_regime` label are excluded from this base rate.

**Hour-matched random-entry baseline**
- For each setup and period: draw, without replacement, as many random eligible window hours from each hour_ny (9, 10) as the setup traded in that hour, and record the mean net return.
- Repeat 2,000 times, with seed 0 for development and seed 1 for test, as in notebook 07.
- Report the median of the random means and the share of random means greater than or equal to the setup's mean.

**Number of comparisons, stated explicitly**
- Base-rate comparisons: 2 setups × 3 base rates × 2 periods = **12**.
- Random-entry baseline comparisons: 2 setups × 1 baseline × 2 periods = **4**.
- Total: **16 comparisons**, 8 on development data and 8 on held-out data.
- As a reference only (not a pass/fail rule): with 8 held-out comparisons, a Bonferroni-adjusted two-sided 5% level corresponds to |difference / SE| > 2.73.

**Limitations known in advance**
- The trading window was chosen using descriptive volatility statistics from 2017–2026, including the 2024+ test period (Gates 1–2). The window is therefore not independent of the test period. That choice used volatility only, never price direction.
- Window hours give at most two trades per day. With setup filters, the number of trades per period may be small; standard errors are reported for that reason.
- The cost assumption (0.15% per side) should be checked against the current exchange fee schedule.

## Frequency check (counts only, development period)

Run after the Gate 3 approval of S1 and S2, before any evaluation. It counts, for development-period window hours (T before 2024-01-01), how many each setup trades and how many are skipped for missing data, split by hour_ny 9 and 10. **No return or outcome is computed:** the exit candle is checked only for existence, never for its price.

- Window hours are listed by clock time: every UTC hour whose start is 09:00 or 10:00 New York time. An hour whose own entry candle is missing is therefore counted as skipped, not silently left out.
- Skipped: the decision candle (T − 15 min), the entry candle (T) or the exit candle (T + 1h) is missing, or a feature the setup uses is NaN on the decision row (as pre-registered).
- Traded: not skipped, and every condition of the setup holds.

In [3]:
import numpy as np

SETUPS = {
    "S1": {"features": ["prev_day_trend", "ret_1h", "rel_volume_1h"],
           "rule": lambda d: (d["prev_day_trend"] == "up") & (d["ret_1h"] > 0) & (d["rel_volume_1h"] > REL_VOLUME_THRESHOLD)},
    "S2": {"features": ["prev_day_trend", "ret_4h", "rv_4h"],
           "rule": lambda d: (d["prev_day_trend"] == "up") & (d["ret_4h"] < 0) & (d["rv_4h"] > RV_4H_THRESHOLD)},
}

clock = pd.date_range(feats.index[0].ceil("h"), feats.index[-1], freq="h").as_unit(feats.index.unit)
clock_hour_ny = clock.tz_convert("America/New_York").hour
is_window = np.isin(clock_hour_ny, WINDOW_HOURS_NY) & (clock < DEV_END)
T = clock[is_window]
hour_ny = pd.Series(clock_hour_ny[is_window], index=T, name="hour_ny")

decision = feats.reindex(T - DECISION_LAG).set_axis(T)
candles_present = (
    T.isin(feats.index + DECISION_LAG)      # decision candle at T - 15 min
    & T.isin(feats.index)                   # entry candle at T
    & T.isin(feats.index - pd.Timedelta("1h"))  # exit candle at T + 1h (existence only)
)

rows = []
for name, setup in SETUPS.items():
    usable = pd.Series(candles_present, index=T) & decision[setup["features"]].notna().all(axis=1)
    traded = usable & setup["rule"](decision).fillna(False).astype(bool)
    for label, mask in [("hour_ny 9", hour_ny == 9), ("hour_ny 10", hour_ny == 10), ("total", hour_ny > -1)]:
        rows.append({"setup": name, "hours": label,
                     "window hours": int(mask.sum()),
                     "skipped (missing data)": int((mask & ~usable).sum()),
                     "evaluable": int((mask & usable).sum()),
                     "traded": int((mask & traded).sum())})

frequency = pd.DataFrame(rows).set_index(["setup", "hours"])
print(f"development window hours: {len(T):,}, {T[0]} -> {T[-1]}")
print(frequency.to_string())

development window hours: 4,656, 2017-08-17 13:00:00+00:00 -> 2023-12-31 15:00:00+00:00
                  window hours  skipped (missing data)  evaluable  traded
setup hours                                                              
S1    hour_ny 9           2328                     208       2120      95
      hour_ny 10          2328                     206       2122      97
      total               4656                     414       4242     192
S2    hour_ny 9           2328                     113       2215     186
      hour_ny 10          2328                     110       2218     196
      total               4656                     223       4433     382


## Decision (Gate 3)

**Decision:** S1 and S2 are approved exactly as pre-registered, with no changes. The frequency check (counts only, no outcomes) is reported above. Evaluation proceeds on the development period only; the held-out test runs once, after a separate decision.

## Development evaluation

The evaluation plan above, run exactly as pre-registered, for the **development period only** (window hours with T on or before 2023-12-31). The rules come from `INTRADAY_SETUPS` and `intraday_setup_flags` in `src/cryptosense/intraday.py`, which contain S1 and S2 as pre-registered, with the printed thresholds.

**Nothing from 2024 onward is computed:** the features are cut to candles opening before 2024-01-01 00:00 UTC before any flag or return is calculated. The last development trade exits at 2023-12-31 16:00 UTC.

**Definitions used (from the pre-registration)**
- Net log return of a trade: log(exit × (1 − 0.0015) / (entry × (1 + 0.0015))). Entry is the open of the candle at T, exit the open of the candle at T + 1h.
- Eligible window hour: the decision candle (T − 15 min), the entry candle (T) and the exit candle (T + 1h) all exist.
- Traded: eligible and the setup fires (its features are present and all conditions hold). Skipped: not eligible, or a feature the setup uses is NaN.
- Base rates use every eligible window hour, at the same cost. The difference in standard errors is (setup mean − base-rate mean) / setup standard error.
- Base rate 3 excludes window hours without a `prev_day_vol_regime` label; its weights come from the setup's trades that have a label.

In [4]:
from cryptosense.intraday import INTRADAY_SETUPS, intraday_setup_flags

COST_PER_SIDE = 0.0015
N_RUNS = 2000
ONE_HOUR = pd.Timedelta("1h")
SETUP_NAMES = list(INTRADAY_SETUPS)

dev_feats = feats[feats.index < DEV_END]          # nothing from 2024 onward
ev = intraday_setup_flags(dev_feats)
ev = ev[ev.index < DEV_END]

entry = dev_feats["open"].reindex(ev.index).to_numpy()
exit_ = dev_feats["open"].reindex(ev.index + ONE_HOUR).to_numpy()
ev["eligible"] = ev.index.isin(dev_feats.index + DECISION_LAG) & ~np.isnan(entry) & ~np.isnan(exit_)
ev["net_log_return"] = np.log(exit_ * (1 - COST_PER_SIDE) / (entry * (1 + COST_PER_SIDE)))
ev["prev_vol_regime"] = dev_feats["prev_day_vol_regime"].reindex(ev.index - DECISION_LAG).to_numpy()
for name in SETUP_NAMES:
    ev[f"{name}_traded"] = ev[name] & ev["eligible"]
    ev[f"{name}_skipped"] = ~(ev[f"{name}_usable"] & ev["eligible"])

print(f"development window hours: {len(ev):,}, {ev.index[0]} -> {ev.index[-1]}")
print(f"eligible window hours: {int(ev['eligible'].sum()):,}")
print(f"last exit: {ev.index[-1] + ONE_HOUR}")

# The package rules must reproduce the Gate 3 frequency check exactly.
for name in SETUP_NAMES:
    assert int(ev[f"{name}_traded"].sum()) == frequency.loc[(name, "total"), "traded"]
    assert int(ev[f"{name}_skipped"].sum()) == frequency.loc[(name, "total"), "skipped (missing data)"]
print("trade and skip counts match the frequency check")

development window hours: 4,656, 2017-08-17 13:00:00+00:00 -> 2023-12-31 15:00:00+00:00
eligible window hours: 4,645
last exit: 2023-12-31 16:00:00+00:00
trade and skip counts match the frequency check


### Setup trades: net 1-hour log return after costs (development)

Mean, median and standard error are in percent (log return × 100). Share positive is the share of trades with a net return above 0.

In [5]:
def measures(r):
    """n, mean, median, share positive and standard error of a set of net log returns (in %)."""
    r = pd.Series(r).dropna()
    return {"n": len(r), "mean_%": 100 * r.mean(), "median_%": 100 * r.median(),
            "share_positive": (r > 0).mean(), "se_%": 100 * r.std() / np.sqrt(len(r))}


rows = []
for name in SETUP_NAMES:
    for label, mask in [("hour_ny 9", ev["hour_ny"] == 9), ("hour_ny 10", ev["hour_ny"] == 10),
                        ("total", ev["hour_ny"] > 0)]:
        traded = ev[mask & ev[f"{name}_traded"]]
        rows.append({"setup": name, "hours": label,
                     "skipped": int((mask & ev[f"{name}_skipped"]).sum()),
                     **measures(traded["net_log_return"])})
setup_table = pd.DataFrame(rows).set_index(["setup", "hours"])
print(setup_table.to_string(formatters={"n": "{:,}".format, "skipped": "{:,}".format,
                                        "share_positive": "{:.3f}".format,
                                        **{c: "{:.4f}".format for c in ["mean_%", "median_%", "se_%"]}}))

                 skipped   n  mean_% median_% share_positive   se_%
setup hours                                                        
S1    hour_ny 9      208  95 -0.2643  -0.2854          0.337 0.1159
      hour_ny 10     206  97 -0.4217  -0.3788          0.299 0.1194
      total          414 192 -0.3438  -0.3310          0.318 0.0832
S2    hour_ny 9      113 186 -0.2564  -0.2658          0.382 0.0869
      hour_ny 10     110 196 -0.3838  -0.3335          0.347 0.1033
      total          223 382 -0.3218  -0.2839          0.364 0.0678


### Base rates (development)

Every eligible window hour is treated as a trade at the same cost. The first table shows the base-rate cells; the second compares each setup with the three pre-registered base rates.

In [6]:
base = ev[ev["eligible"]]
cells_hour = base.groupby("hour_ny")["net_log_return"].agg(n="size", mean="mean")
cells_regime = (base.dropna(subset=["prev_vol_regime"])
                .groupby(["hour_ny", "prev_vol_regime"])["net_log_return"].agg(n="size", mean="mean"))

print("All eligible window hours:", f"n = {len(base):,}, mean = {100 * base['net_log_return'].mean():.4f}%")
print("\nBy hour_ny:")
print(cells_hour.assign(mean=100 * cells_hour["mean"]).rename(columns={"mean": "mean_%"})
      .to_string(float_format="{:.4f}".format))
print("\nBy hour_ny and previous-day volatility regime:")
print(cells_regime.assign(mean=100 * cells_regime["mean"]).rename(columns={"mean": "mean_%"})
      .to_string(float_format="{:.4f}".format))

All eligible window hours: n = 4,645, mean = -0.3089%

By hour_ny:
            n  mean_%
hour_ny              
9        2322 -0.2946
10       2323 -0.3231

By hour_ny and previous-day volatility regime:
                            n  mean_%
hour_ny prev_vol_regime              
9       high              867 -0.2822
        low              1072 -0.3106
10      high              868 -0.3228
        low              1072 -0.3138


In [7]:
rows = []
for name in SETUP_NAMES:
    trades = ev[ev[f"{name}_traded"]]
    m = measures(trades["net_log_return"])
    mean, se = m["mean_%"] / 100, m["se_%"] / 100

    b1 = base["net_log_return"].mean()
    w_hour = trades["hour_ny"].value_counts(normalize=True)
    b2 = sum(w * cells_hour.loc[h, "mean"] for h, w in w_hour.items())
    labelled = trades.dropna(subset=["prev_vol_regime"])
    w_cell = labelled.groupby(["hour_ny", "prev_vol_regime"]).size() / len(labelled)
    b3 = sum(w * cells_regime.loc[cell, "mean"] for cell, w in w_cell.items())

    for label, b, note in [("1. all window hours", b1, ""),
                           ("2. same window hour", b2, ""),
                           ("3. same hour and prev-day vol regime", b3,
                            f"weights from {len(labelled)} of {len(trades)} trades with a label")]:
        rows.append({"setup": name, "base rate": label, "setup_mean_%": 100 * mean, "setup_se_%": 100 * se,
                     "base_mean_%": 100 * b, "diff_%": 100 * (mean - b), "diff_in_se": (mean - b) / se,
                     "note": note})
base_table = pd.DataFrame(rows).set_index(["setup", "base rate"])
print(base_table.to_string(float_format="{:.4f}".format))

                                            setup_mean_%  setup_se_%  base_mean_%  diff_%  diff_in_se                                         note
setup base rate                                                                                                                                   
S1    1. all window hours                        -0.3438      0.0832      -0.3089 -0.0349     -0.4201                                             
      2. same window hour                        -0.3438      0.0832      -0.3090 -0.0348     -0.4184                                             
      3. same hour and prev-day vol regime       -0.3438      0.0832      -0.3083 -0.0355     -0.4267  weights from 176 of 192 trades with a label
S2    1. all window hours                        -0.3218      0.0678      -0.3089 -0.0129     -0.1903                                             
      2. same window hour                        -0.3218      0.0678      -0.3092 -0.0125     -0.1848                 

### Hour-matched random-entry baseline (development, 2,000 draws, seed 0)

For each setup: draw, without replacement, as many random eligible window hours from each hour_ny as the setup traded in that hour, and record the mean net return. Repeated 2,000 times with one generator seeded 0, used for S1 and then S2.

In [8]:
rng = np.random.default_rng(0)
pools = {h: base.loc[base["hour_ny"] == h, "net_log_return"].to_numpy() for h in WINDOW_HOURS_NY}

rows = []
for name in SETUP_NAMES:
    trades = ev[ev[f"{name}_traded"]]
    k = trades["hour_ny"].value_counts()
    actual = trades["net_log_return"].mean()
    sims = np.array([
        np.concatenate([rng.choice(pools[h], size=int(k.get(h, 0)), replace=False)
                        for h in WINDOW_HOURS_NY]).mean()
        for _ in range(N_RUNS)
    ])
    rows.append({"setup": name, "trades_h9": int(k.get(9, 0)), "trades_h10": int(k.get(10, 0)),
                 "setup_mean_%": 100 * actual, "random_median_%": 100 * np.median(sims),
                 "share_random_ge_setup": (sims >= actual).mean()})
random_table = pd.DataFrame(rows).set_index("setup")
print(random_table.to_string(float_format="{:.4f}".format))

       trades_h9  trades_h10  setup_mean_%  random_median_%  share_random_ge_setup
setup                                                                             
S1            95          97       -0.3438          -0.3107                 0.6975
S2           186         196       -0.3218          -0.3100                 0.5950


## Development results

Development period only: window hours with T from 2017-08-17 13:00 to 2023-12-31 15:00 UTC (4,656 window hours, 4,645 eligible). Net 1-hour log returns after 0.15% costs per side, in %. Difference in SE = (setup mean − base-rate mean) / setup SE. The package rules reproduce the Gate 3 frequency check exactly (S1 192 trades, 414 skipped; S2 382 trades, 223 skipped).

### Setup trades

| Setup | Hours | Skipped | Trades | Mean | Median | Share positive | SE |
|---|---|---|---|---|---|---|---|
| S1 | hour_ny 9 | 208 | 95 | −0.2643 | −0.2854 | 0.337 | 0.1159 |
| S1 | hour_ny 10 | 206 | 97 | −0.4217 | −0.3788 | 0.299 | 0.1194 |
| S1 | total | 414 | 192 | −0.3438 | −0.3310 | 0.318 | 0.0832 |
| S2 | hour_ny 9 | 113 | 186 | −0.2564 | −0.2658 | 0.382 | 0.0869 |
| S2 | hour_ny 10 | 110 | 196 | −0.3838 | −0.3335 | 0.347 | 0.1033 |
| S2 | total | 223 | 382 | −0.3218 | −0.2839 | 0.364 | 0.0678 |

### Base-rate cells (all eligible window hours)

| Cell | n | Mean |
|---|---|---|
| All window hours | 4,645 | −0.3089 |
| hour_ny 9 | 2,322 | −0.2946 |
| hour_ny 10 | 2,323 | −0.3231 |
| hour_ny 9, prev-day high | 867 | −0.2822 |
| hour_ny 9, prev-day low | 1,072 | −0.3106 |
| hour_ny 10, prev-day high | 868 | −0.3228 |
| hour_ny 10, prev-day low | 1,072 | −0.3138 |

### Comparisons with base rates

| Setup | Base rate | Setup mean | Setup SE | Base mean | Difference | Difference in SE |
|---|---|---|---|---|---|---|
| S1 | 1. all window hours | −0.3438 | 0.0832 | −0.3089 | −0.0349 | −0.42 |
| S1 | 2. same window hour | −0.3438 | 0.0832 | −0.3090 | −0.0348 | −0.42 |
| S1 | 3. same hour and prev-day vol regime | −0.3438 | 0.0832 | −0.3083 | −0.0355 | −0.43 |
| S2 | 1. all window hours | −0.3218 | 0.0678 | −0.3089 | −0.0129 | −0.19 |
| S2 | 2. same window hour | −0.3218 | 0.0678 | −0.3092 | −0.0125 | −0.18 |
| S2 | 3. same hour and prev-day vol regime | −0.3218 | 0.0678 | −0.3062 | −0.0155 | −0.23 |

Base rate 3 weights come from the trades with a `prev_day_vol_regime` label: S1 176 of 192, S2 325 of 382.

### Hour-matched random-entry baseline (2,000 draws, seed 0)

| Setup | Trades (h9 / h10) | Setup mean | Random median | Share of random means ≥ setup mean |
|---|---|---|---|---|
| S1 | 95 / 97 | −0.3438 | −0.3107 | 0.6975 |
| S2 | 186 / 196 | −0.3218 | −0.3100 | 0.5950 |

## Decision (Gate 4)

**Decision:** The held-out test is not run. In the development period, neither setup differed from any base rate by more than 0.43 standard errors, and 60–70% of hour-matched random entries did as well. The average window hour returned −0.31% after costs, close to the 0.30% round-trip cost, so gross 1-hour moves averaged about zero and only direction-picking skill could produce a profit; neither setup showed any. With no development result to confirm, the 2024 onward period is kept unseen for future hypotheses. S1 and S2 are closed as negative results, and Phase 11 is not needed for them.